In [1]:
import pandas as pd
import numpy as np

# Charger le dataset
df = pd.read_csv("data.csv")

# Supprimer les colonnes inutiles
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Transformer la cible :
# M = Malignant -> 1
# B = Benign -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Séparer X et y
X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(int)

#Train / Test split
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]




In [2]:
# Fonctions utiles

# Calcule l'impureté de Gini.
 #   Si un groupe contient une seule classe, Gini = 0.
  #  Si les classes sont mélangées, Gini augmente.
def gini(y):
  
    classes = np.unique(y)
    impurity = 1

    for c in classes:
        p = np.sum(y == c) / len(y)
        impurity -= p ** 2

    return impurity
    
 # Divise le dataset en deux parties :
 #   - gauche : valeurs <= seuil
  #  - droite : valeurs > seuil

def split_dataset(X, y, feature_index, threshold):
    
    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, y_left, X_right, y_right

#Cherche la meilleure variable et le meilleur seuil
# pour diviser les données.
def best_split(X, y):
    
    best_feature = None
    best_threshold = None
    best_gini = 1

    n_samples, n_features = X.shape

    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])

        for threshold in thresholds:
            X_left, y_left, X_right, y_right = split_dataset(
                X, y, feature_index, threshold
            )

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gini_left = gini(y_left)
            gini_right = gini(y_right)

            weighted_gini = (
                len(y_left) / n_samples * gini_left
                + len(y_right) / n_samples * gini_right
            )

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature_index
                best_threshold = threshold

    return best_feature, best_threshold

# Retourne la classe la plus fréquente.

def majority_class(y):
    
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]


# Classe Node
#Un noeud de l'arbre.
    #Il peut être :
   # - un noeud interne : contient feature, threshold, left, right
   #- une feuille : contient une classe finale
class Node:

    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value




In [3]:
#  Arbre de décision from scratch
class DecisionTreeFromScratch:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
#Fonction d'entraînement.
# Elle construit l'arbre à partir des données.
    def fit(self, X, y):
        self.root = self.build_tree(X, y, depth=0)
#Construction récursive de l'arbre.
    def build_tree(self, X, y, depth):

        n_samples = len(y)
        n_classes = len(np.unique(y))

        # Conditions d'arrêt :
        # 1. profondeur maximale atteinte
        # 2. une seule classe restante
        # 3. pas assez d'exemples pour diviser
        if (
            depth >= self.max_depth
            or n_classes == 1
            or n_samples < self.min_samples_split
        ):
            leaf_value = majority_class(y)
            return Node(value=leaf_value)

        # Chercher la meilleure séparation
        feature, threshold = best_split(X, y)

        # Si aucune séparation trouvée
        if feature is None:
            leaf_value = majority_class(y)
            return Node(value=leaf_value)

        # Diviser les données
        X_left, y_left, X_right, y_right = split_dataset(X, y, feature, threshold)

        # Construire les sous-arbres
        left_child = self.build_tree(X_left, y_left, depth + 1)
        right_child = self.build_tree(X_right, y_right, depth + 1)

        return Node(feature=feature, threshold=threshold, left=left_child, right=right_child)
#  Prédit la classe d'une seule observation.
    def predict_one(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left)
        else:
            return self.predict_one(x, node.right)
# Prédit les classes de plusieurs observations.
    def predict(self, X):
        
        return np.array([self.predict_one(x, self.root) for x in X])


# Entraîner le modèle
tree = DecisionTreeFromScratch(max_depth=5, min_samples_split=2)
tree.fit(X_train, y_train)

#  Prédictions
y_pred = tree.predict(X_test)

# Évaluation simple 
correct = 0
incorrect = 0

for i in range(len(y_test)):
    if y_test[i] == y_pred[i]:
        correct += 1
    else:
        incorrect += 1

print("Évaluation du modèle :")
print("Bonnes prédictions :", correct)
print("Mauvaises prédictions :", incorrect)
print("Total :", len(y_test))


Évaluation du modèle :
Bonnes prédictions : 108
Mauvaises prédictions : 6
Total : 114


In [4]:
# 10) Affichage de l'arbre de décision

def print_tree(node, depth=0):
    indent = "  " * depth

    # Si feuille (classe finale)
    if node.value is not None:
        label = "Malignant" if node.value == 1 else "Benign"
        print(indent + "→ Classe :", label)
        return

    # Sinon afficher la règle
    print(indent + f"Si X[{node.feature}] <= {round(node.threshold, 3)} :")

    # Sous-arbre gauche
    print(indent + "Gauche :")
    print_tree(node.left, depth + 1)

    # Sous-arbre droit
    print(indent + "Droite :")
    print_tree(node.right, depth + 1)

In [5]:
print("\n Structure de l'arbre :\n")
print_tree(tree.root)


 Structure de l'arbre :

Si X[23] <= 880.8 :
Gauche :
  Si X[27] <= 0.16 :
  Gauche :
    Si X[27] <= 0.132 :
    Gauche :
      Si X[10] <= 0.881 :
      Gauche :
        Si X[13] <= 37.83 :
        Gauche :
          → Classe : Benign
        Droite :
          → Classe : Benign
      Droite :
        → Classe : Malignant
    Droite :
      Si X[1] <= 20.22 :
      Gauche :
        Si X[23] <= 809.8 :
        Gauche :
          → Classe : Benign
        Droite :
          → Classe : Malignant
      Droite :
        → Classe : Malignant
  Droite :
    Si X[21] <= 23.19 :
    Gauche :
      → Classe : Benign
    Droite :
      → Classe : Malignant
Droite :
  Si X[6] <= 0.071 :
  Gauche :
    Si X[1] <= 19.46 :
    Gauche :
      Si X[5] <= 0.056 :
      Gauche :
        → Classe : Malignant
      Droite :
        → Classe : Benign
    Droite :
      → Classe : Malignant
  Droite :
    → Classe : Malignant
